# UHPC App Validation

**Author:** K Flowers  
**Purpose:** Validate that the Streamlit app at `app/` produces the same numeric outputs (strength prediction, SHAP values, CO₂ emissions) as the trained model artifacts and documented emission factors.

This notebook is **read-only** with respect to the project — it does not refit the model, modify any artifacts, or write any files. It is intended to be re-run anytime as a regression check for the app.

**Pipeline Position:** Notebook 4 of 4 — App Validation
- 01_exploratory_analysis.ipynb
- 02_model_development.ipynb
- 03_model_interpretation.ipynb
- 04_app_validation.ipynb ← this notebook

**Sections:**
1. Setup and Artifact Loading
2. Reproducibility Tests (low / mid / high strength rows)
3. Edge Case Inputs
4. SHAP Value Comparison
5. CO₂ Calculation Validation
6. Summary

---
## 1. Setup and Artifact Loading

Loads the same model artifact, training data (SHAP background), and emission factor table that `app/app.py` uses. Verifies file paths match those defined in `app/shared.py` before proceeding.

In [1]:
# Standard library
import sys
from pathlib import Path

# Core data libraries
import numpy as np
import pandas as pd
import joblib
import shap

# Add app/ to sys.path so we can import the app's CO2 module
PROJECT_ROOT = Path.cwd().parent
APP_DIR = PROJECT_ROOT / 'app'
if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))

from emission_factors import compute_co2, EMISSION_FACTORS, NON_MATERIAL_FEATURES

print('Imports loaded successfully.')

Imports loaded successfully.


In [2]:
# Resolve artifact paths and verify they match what app/shared.py uses
RESULTS_DIR = PROJECT_ROOT / 'data' / 'results'
MODEL_PATH = RESULTS_DIR / 'xgb_tuned_model.joblib'
PROCESSED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'uhpc_dataset_cleaned.csv'
X_TRAIN_PATH = RESULTS_DIR / 'X_train.csv'

# Mirror the path resolution in app/shared.py
SHARED_PROJECT_ROOT = APP_DIR.parent
SHARED_MODEL_PATH = SHARED_PROJECT_ROOT / 'data' / 'results' / 'xgb_tuned_model.joblib'

assert MODEL_PATH == SHARED_MODEL_PATH, (
    f'Path mismatch with app/shared.py:\n  notebook: {MODEL_PATH}\n  shared:   {SHARED_MODEL_PATH}'
)
assert MODEL_PATH.exists(), f'Model file not found at {MODEL_PATH}'
assert PROCESSED_PATH.exists(), f'Processed dataset not found at {PROCESSED_PATH}'
assert X_TRAIN_PATH.exists(), f'X_train not found at {X_TRAIN_PATH}'

# Constant mirrored from app/shared.py
UHPC_THRESHOLD = 150  # MPa

print('Model path:    ', MODEL_PATH.relative_to(PROJECT_ROOT))
print('X_train path:  ', X_TRAIN_PATH.relative_to(PROJECT_ROOT))
print('Processed path:', PROCESSED_PATH.relative_to(PROJECT_ROOT))
print('UHPC threshold:', UHPC_THRESHOLD, 'MPa')
print('Path verification passed -- paths match app/shared.py.')

Model path:     data\results\xgb_tuned_model.joblib
X_train path:   data\results\X_train.csv
Processed path: data\processed\uhpc_dataset_cleaned.csv
UHPC threshold: 150 MPa
Path verification passed -- paths match app/shared.py.


In [3]:
# Load the model artifact (same call pattern as app/shared.load_model)
artifacts = joblib.load(MODEL_PATH)
model = artifacts['model']
feature_names = artifacts['feature_names']

# Load SHAP background data (same as app uses)
X_train = pd.read_csv(X_TRAIN_PATH)

# Build the SHAP explainer matching the app's pattern
explainer = shap.TreeExplainer(model, data=X_train)

# Load the cleaned dataset for spot-check rows
df = pd.read_csv(PROCESSED_PATH)

print(f'Model type:           {type(model).__name__}')
print(f'Feature names ({len(feature_names)}):')
for i, f in enumerate(feature_names):
    print(f'  {i+1:2d}. {f}')
print(f'\nTraining records:     {len(X_train)}')
print(f'Processed dataset:    {len(df)} records')
print(f'\nEmission factor keys ({len(EMISSION_FACTORS)}):')
for k in EMISSION_FACTORS:
    print(f'  - {k}')
print(f'\nNon-material features (skipped by compute_co2): {sorted(NON_MATERIAL_FEATURES)}')

Model type:           XGBRegressor
Feature names (13):
   1. cement
   2. slag
   3. silica_fume
   4. limestone_powder
   5. quartz_powder
   6. fly_ash
   7. nano_silica
   8. aggregate
   9. water
  10. fiber
  11. superplasticizer
  12. temperature
  13. age

Training records:     633
Processed dataset:    792 records

Emission factor keys (12):
  - cement
  - water
  - fly_ash
  - silica_fume
  - nano_silica
  - quartz_powder
  - limestone_powder
  - slag
  - superplasticizer
  - aggregate
  - fiber_virgin
  - fiber_recycled

Non-material features (skipped by compute_co2): ['age', 'temperature']


---
## 2. Reproducibility Tests

Selects three observations from the processed dataset — one low-strength, one mid-strength, one high-strength — and runs each through the model. The printed input values can be entered manually into the app sliders to confirm the app returns the same prediction for the same inputs.

**Note on slider granularity:** The Streamlit sliders use feature-specific step sizes (e.g., 10 kg/m³ for cement). Inputs may need to be rounded to slider granularity when entering them by hand.

In [4]:
# Pick three rows by strength percentile from the cleaned dataset
df_sorted = df.sort_values('compressive_strength').reset_index(drop=True)
n = len(df_sorted)

sample_indices = {
    'low':  n // 20,            # ~5th percentile
    'mid':  n // 2,             # ~50th percentile
    'high': n - 1 - (n // 20),  # ~95th percentile
}

samples = {label: df_sorted.iloc[idx] for label, idx in sample_indices.items()}

for label, row in samples.items():
    print(f'\n--- {label.upper()} STRENGTH SAMPLE (row index {sample_indices[label]}) ---')
    print('Inputs (enter these into the app sliders):')
    for f in feature_names:
        print(f'  {f:20s}: {row[f]}')
    print(f'Measured strength:  {row["compressive_strength"]:.2f} MPa')


--- LOW STRENGTH SAMPLE (row index 39) ---
Inputs (enter these into the app sliders):
  cement              : 545.875
  slag                : 0.0
  silica_fume         : 0.0
  limestone_powder    : 0.0
  quartz_powder       : 0.0
  fly_ash             : 0.0
  nano_silica         : 4.125
  aggregate           : 1685.0
  water               : 192.5
  fiber               : 0.0
  superplasticizer    : 4.4
  temperature         : 21.0
  age                 : 28.0
Measured strength:  54.50 MPa

--- MID STRENGTH SAMPLE (row index 396) ---
Inputs (enter these into the app sliders):
  cement              : 472.0
  slag                : 315.0
  silica_fume         : 262.0
  limestone_powder    : 0.0
  quartz_powder       : 0.0
  fly_ash             : 0.0
  nano_silica         : 0.0
  aggregate           : 1049.0
  water               : 178.0
  fiber               : 156.0
  superplasticizer    : 21.0
  temperature         : 20.0
  age                 : 7.0
Measured strength:  120.90 MPa

--- HIG

In [5]:
# Run each sample through the model and report prediction + abs error
repro_results = []

for label, row in samples.items():
    input_df = pd.DataFrame([{f: row[f] for f in feature_names}])
    pred = float(model.predict(input_df)[0])
    measured = float(row['compressive_strength'])
    abs_err = abs(measured - pred)

    repro_results.append({
        'Sample':    label,
        'Measured':  round(measured, 2),
        'Predicted': round(pred, 2),
        'Abs Error': round(abs_err, 2),
    })

repro_df = pd.DataFrame(repro_results)
print('Reproducibility summary (model predictions for known rows):')
print(repro_df.to_string(index=False))

REPRO_PASSED = int((repro_df['Abs Error'] < 30).sum())  # generous bound (~5x test RMSE)
print(f'\nRows with abs error < 30 MPa: {REPRO_PASSED} of {len(repro_df)}')

Reproducibility summary (model predictions for known rows):
Sample  Measured  Predicted  Abs Error
   low      54.5      55.16       0.66
   mid     120.9     114.72       6.18
  high     187.5     189.54       2.04

Rows with abs error < 30 MPa: 3 of 3


---
## 3. Edge Case Inputs

Five hand-crafted mixes covering the input range. For each, the model is evaluated and the top three SHAP features by absolute impact are reported. Confirms the app handles boundary conditions (max age, min age, zero supplementary materials, cement-heavy, SCM-heavy).

In [6]:
# Default mix -- values mirror the FEATURE_CONFIG defaults in app/shared.py
def default_mix():
    return {
        'cement': 770.0, 'slag': 0.0, 'silica_fume': 144.0,
        'limestone_powder': 0.0, 'quartz_powder': 0.0, 'fly_ash': 0.0,
        'nano_silica': 0.0, 'aggregate': 1104.0, 'water': 177.0,
        'fiber': 0.0, 'superplasticizer': 30.0,
        'temperature': 21.0, 'age': 28.0,
    }

edge_mixes = {
    'high_cement':       {**default_mix(), 'cement': 1200.0, 'silica_fume': 0.0},
    'high_scm':          {**default_mix(), 'cement': 500.0, 'slag': 200.0,
                          'silica_fume': 200.0, 'fly_ash': 100.0},
    'max_age':           {**default_mix(), 'age': 365.0},
    'min_age':           {**default_mix(), 'age': 1.0},
    'no_supplementary':  {**default_mix(), 'silica_fume': 0.0, 'slag': 0.0,
                          'fly_ash': 0.0, 'limestone_powder': 0.0,
                          'quartz_powder': 0.0, 'nano_silica': 0.0,
                          'fiber': 0.0},
}

print('Edge mixes defined:')
for label, mix in edge_mixes.items():
    print(f'  {label}')

Edge mixes defined:
  high_cement
  high_scm
  max_age
  min_age
  no_supplementary


In [7]:
# Predict + SHAP top-3 for each edge mix
edge_results = []

for label, mix in edge_mixes.items():
    input_df = pd.DataFrame([{f: mix[f] for f in feature_names}])
    pred = float(model.predict(input_df)[0])
    classification = 'Meets UHPC' if pred >= UHPC_THRESHOLD else 'Below UHPC'

    shap_explanation = explainer(input_df)
    shap_values = shap_explanation.values[0]
    abs_impact = np.abs(shap_values)
    top3_idx = np.argsort(abs_impact)[::-1][:3]
    top3 = [
        f'{feature_names[i]} ({shap_values[i]:+.2f})' for i in top3_idx
    ]

    edge_results.append({
        'Mix Label':       label,
        'Predicted (MPa)': round(pred, 2),
        'Classification':  classification,
        'Top 3 SHAP':      ' | '.join(top3),
    })

edge_df = pd.DataFrame(edge_results)
print('Edge case predictions:')
print(edge_df.to_string(index=False))

EDGE_PASSED = int(edge_df['Predicted (MPa)'].notna().sum())
print(f'\nEdge predictions completed without error: {EDGE_PASSED} of {len(edge_df)}')

Edge case predictions:
       Mix Label  Predicted (MPa) Classification                                          Top 3 SHAP
     high_cement           113.06     Below UHPC      age (+12.26) | fiber (-11.18) | cement (+8.06)
        high_scm           114.86     Below UHPC age (+17.63) | fiber (-11.37) | silica_fume (+9.66)
         max_age           132.77     Below UHPC        age (+32.68) | fiber (-8.38) | water (-3.87)
         min_age            65.24     Below UHPC    age (-43.77) | fiber (-9.33) | aggregate (-3.63)
no_supplementary           111.55     Below UHPC age (+13.19) | fiber (-11.00) | silica_fume (-6.46)

Edge predictions completed without error: 5 of 5


---
## 4. SHAP Value Comparison

For two of the rows from the reproducibility section (low and high strength), prints the full SHAP value array and verifies the SHAP additivity property:

$$
\text{prediction} \approx \text{base value} + \sum_i \text{SHAP}_i
$$

Each per-feature SHAP value can be cross-checked against the SHAP impact column the app displays for the same input.

In [8]:
# Print full SHAP arrays for the low and high samples
shap_check_results = []

for label in ['low', 'high']:
    row = samples[label]
    input_df = pd.DataFrame([{f: row[f] for f in feature_names}])
    pred = float(model.predict(input_df)[0])

    shap_explanation = explainer(input_df)
    shap_values = shap_explanation.values[0]

    base_value = float(
        shap_explanation.base_values[0]
        if hasattr(shap_explanation, 'base_values') and shap_explanation.base_values is not None
        else explainer.expected_value
    )

    sum_shap = float(np.sum(shap_values))
    reconstructed = base_value + sum_shap
    diff = abs(reconstructed - pred)
    additivity_ok = diff < 1e-3

    print(f'\n--- SHAP for {label.upper()} sample ---')
    print(f'Base value (expected_value): {base_value:.4f} MPa')
    print('Per-feature SHAP values (compare to app SHAP table):')
    for f, v in zip(feature_names, shap_values):
        print(f'  {f:20s}: {v:+.4f}')
    print(f'Sum of SHAP values:    {sum_shap:+.4f}')
    print(f'Base + sum:            {reconstructed:.4f}')
    print(f'Model prediction:      {pred:.4f}')
    print(f'Reconstruction diff:   {diff:.6f}  (<1e-3 = pass)')

    shap_check_results.append({
        'Sample':           label,
        'Prediction':       round(pred, 4),
        'Base + Sum SHAP':  round(reconstructed, 4),
        'Diff':             round(diff, 6),
        'Additivity OK':    additivity_ok,
    })

shap_check_df = pd.DataFrame(shap_check_results)
print('\n--- SHAP additivity check ---')
print(shap_check_df.to_string(index=False))
SHAP_PASSED = int(shap_check_df['Additivity OK'].sum())


--- SHAP for LOW sample ---
Base value (expected_value): 116.4440 MPa
Per-feature SHAP values (compare to app SHAP table):
  cement              : -15.8796
  slag                : -0.0380
  silica_fume         : -19.7453
  limestone_powder    : -0.0902
  quartz_powder       : -0.1005
  fly_ash             : -1.3105
  nano_silica         : -0.8143
  aggregate           : -11.2947
  water               : -1.9823
  fiber               : -11.4292
  superplasticizer    : -10.1903
  temperature         : -1.1271
  age                 : +12.7166
Sum of SHAP values:    -61.2853
Base + sum:            55.1587
Model prediction:      55.1588
Reconstruction diff:   0.000029  (<1e-3 = pass)

--- SHAP for HIGH sample ---
Base value (expected_value): 116.4440 MPa
Per-feature SHAP values (compare to app SHAP table):
  cement              : +6.2225
  slag                : -0.0802
  silica_fume         : +12.8901
  limestone_powder    : -0.0632
  quartz_powder       : +0.3405
  fly_ash             : -0

---
## 5. CO₂ Calculation Validation

Three test mixes are run through both:

1. A **manual transparent calculation** that writes out `mass × emission_factor` for every contributing ingredient with the actual numbers visible.
2. The app's `compute_co2()` function from `app/emission_factors.py`.

The two values are compared. Difference should be < 0.01 kg CO₂/m³ (floating-point tolerance).

**Emission factor source:** `docs/emission_factors_v2.md`. Factor tuples are `(default, low, high)` in kg CO₂e/kg material.

**Note:** Non-material features (`temperature`, `age`) are ignored by `compute_co2`. Ingredients with zero mass are skipped from the breakdown.

In [9]:
# Print the EMISSION_FACTORS table for traceability
print('EMISSION_FACTORS (kg CO2e per kg of material) -- (default, low, high):\n')
for k, v in EMISSION_FACTORS.items():
    print(f'  {k:20s}: default={v[0]:.4f}  low={v[1]:.4f}  high={v[2]:.4f}')

# Sanity: low <= default <= high for every entry
for k, (default, low, high) in EMISSION_FACTORS.items():
    assert low <= default <= high, f'Range violation for {k}: low={low}, default={default}, high={high}'
print('\nAll factors satisfy low <= default <= high.')

EMISSION_FACTORS (kg CO2e per kg of material) -- (default, low, high):

  cement              : default=0.8300  low=0.7400  high=0.9500
  water               : default=0.0002  low=0.0001  high=0.0008
  fly_ash             : default=0.0090  low=0.0040  high=0.0500
  silica_fume         : default=0.0140  low=0.0140  high=0.1430
  nano_silica         : default=1.5000  low=0.5000  high=3.0000
  quartz_powder       : default=0.0237  low=0.0160  high=0.0480
  limestone_powder    : default=0.0170  low=0.0070  high=0.0320
  slag                : default=0.0520  low=0.0190  high=0.0830
  superplasticizer    : default=0.9440  low=0.7200  high=2.2000
  aggregate           : default=0.0050  low=0.0030  high=0.0260
  fiber_virgin        : default=2.5000  low=1.9000  high=3.2000
  fiber_recycled      : default=0.9000  low=0.5000  high=1.3000

All factors satisfy low <= default <= high.


In [10]:
# Test mix 1: high cement, low SCM
co2_mix_1 = {
    'cement': 800.0, 'slag': 0.0, 'silica_fume': 0.0,
    'limestone_powder': 0.0, 'quartz_powder': 0.0, 'fly_ash': 0.0,
    'nano_silica': 0.0, 'aggregate': 1100.0, 'water': 180.0,
    'fiber': 0.0, 'superplasticizer': 30.0,
    'temperature': 21.0, 'age': 28.0,
}

# Manual calc -- write out each contributing ingredient with actual numbers
manual_1 = (
    800.0 * 0.830 +     # cement
    1100.0 * 0.005 +    # aggregate
    180.0 * 0.000196 +  # water
    30.0 * 0.944        # superplasticizer
)
# slag, silica_fume, limestone, quartz, fly_ash, nano_silica, fiber: zero -> skipped
# temperature, age: non-material -> skipped

app_1 = compute_co2(co2_mix_1, fiber_type='virgin')['totals']['default']
diff_1 = abs(manual_1 - app_1)
passed_1 = diff_1 < 0.01

print('--- CO2 test mix 1: high cement, low SCM ---')
print('Manual calc:')
print(f'   800.0 * 0.830       = {800.0 * 0.830:.4f}')
print(f'  1100.0 * 0.005       = {1100.0 * 0.005:.4f}')
print(f'   180.0 * 0.000196    = {180.0 * 0.000196:.4f}')
print(f'    30.0 * 0.944       = {30.0 * 0.944:.4f}')
print(f'                 total = {manual_1:.4f}')
print(f'compute_co2 default:   = {app_1:.4f}')
print(f'Diff:                  = {diff_1:.6f}')
print('PASS' if passed_1 else 'FAIL')

--- CO2 test mix 1: high cement, low SCM ---
Manual calc:
   800.0 * 0.830       = 664.0000
  1100.0 * 0.005       = 5.5000
   180.0 * 0.000196    = 0.0353
    30.0 * 0.944       = 28.3200
                 total = 697.8553
compute_co2 default:   = 697.8553
Diff:                  = 0.000000
PASS


In [11]:
# Test mix 2: high SCM (slag + silica fume + fly ash)
co2_mix_2 = {
    'cement': 500.0, 'slag': 200.0, 'silica_fume': 150.0,
    'limestone_powder': 0.0, 'quartz_powder': 0.0, 'fly_ash': 100.0,
    'nano_silica': 0.0, 'aggregate': 1100.0, 'water': 180.0,
    'fiber': 0.0, 'superplasticizer': 30.0,
    'temperature': 21.0, 'age': 28.0,
}

manual_2 = (
    500.0 * 0.830 +     # cement
    200.0 * 0.052 +     # slag
    150.0 * 0.014 +     # silica_fume
    100.0 * 0.009 +     # fly_ash
    1100.0 * 0.005 +    # aggregate
    180.0 * 0.000196 +  # water
    30.0 * 0.944        # superplasticizer
)

app_2 = compute_co2(co2_mix_2, fiber_type='virgin')['totals']['default']
diff_2 = abs(manual_2 - app_2)
passed_2 = diff_2 < 0.01

print('--- CO2 test mix 2: high SCM ---')
print('Manual calc:')
print(f'   500.0 * 0.830       = {500.0 * 0.830:.4f}')
print(f'   200.0 * 0.052       = {200.0 * 0.052:.4f}')
print(f'   150.0 * 0.014       = {150.0 * 0.014:.4f}')
print(f'   100.0 * 0.009       = {100.0 * 0.009:.4f}')
print(f'  1100.0 * 0.005       = {1100.0 * 0.005:.4f}')
print(f'   180.0 * 0.000196    = {180.0 * 0.000196:.4f}')
print(f'    30.0 * 0.944       = {30.0 * 0.944:.4f}')
print(f'                 total = {manual_2:.4f}')
print(f'compute_co2 default:   = {app_2:.4f}')
print(f'Diff:                  = {diff_2:.6f}')
print('PASS' if passed_2 else 'FAIL')

--- CO2 test mix 2: high SCM ---
Manual calc:
   500.0 * 0.830       = 415.0000
   200.0 * 0.052       = 10.4000
   150.0 * 0.014       = 2.1000
   100.0 * 0.009       = 0.9000
  1100.0 * 0.005       = 5.5000
   180.0 * 0.000196    = 0.0353
    30.0 * 0.944       = 28.3200
                 total = 462.2553
compute_co2 default:   = 462.2553
Diff:                  = 0.000000
PASS


In [12]:
# Test mix 3: minimal -- only cement, water, aggregate
# Verifies compute_co2 correctly skips zero-mass and non-material features
co2_mix_3 = {
    'cement': 600.0, 'slag': 0.0, 'silica_fume': 0.0,
    'limestone_powder': 0.0, 'quartz_powder': 0.0, 'fly_ash': 0.0,
    'nano_silica': 0.0, 'aggregate': 1000.0, 'water': 200.0,
    'fiber': 0.0, 'superplasticizer': 0.0,
    'temperature': 21.0, 'age': 28.0,
}

manual_3 = (
    600.0 * 0.830 +     # cement
    1000.0 * 0.005 +    # aggregate
    200.0 * 0.000196    # water
)
# Everything else either zero mass or non-material -> skipped

app_3_result = compute_co2(co2_mix_3, fiber_type='virgin')
app_3 = app_3_result['totals']['default']
diff_3 = abs(manual_3 - app_3)
passed_3 = diff_3 < 0.01

print('--- CO2 test mix 3: minimal mix (cement + water + aggregate) ---')
print('Manual calc:')
print(f'   600.0 * 0.830       = {600.0 * 0.830:.4f}')
print(f'  1000.0 * 0.005       = {1000.0 * 0.005:.4f}')
print(f'   200.0 * 0.000196    = {200.0 * 0.000196:.4f}')
print(f'                 total = {manual_3:.4f}')
print(f'compute_co2 default:   = {app_3:.4f}')
print(f'Diff:                  = {diff_3:.6f}')
print('\nIngredients tracked by compute_co2 (zero-mass and non-material features should be absent):')
for ing in app_3_result['by_ingredient']:
    print(f'  - {ing}')
print('\nPASS' if passed_3 else '\nFAIL')

--- CO2 test mix 3: minimal mix (cement + water + aggregate) ---
Manual calc:
   600.0 * 0.830       = 498.0000
  1000.0 * 0.005       = 5.0000
   200.0 * 0.000196    = 0.0392
                 total = 503.0392
compute_co2 default:   = 503.0392
Diff:                  = 0.000000

Ingredients tracked by compute_co2 (zero-mass and non-material features should be absent):
  - cement
  - aggregate
  - water

PASS


In [13]:
# Roll up CO2 results
co2_results = pd.DataFrame([
    {'Mix': 'high_cement_low_scm', 'Manual': round(manual_1, 4),
     'compute_co2': round(app_1, 4), 'Diff': round(diff_1, 6), 'Pass': passed_1},
    {'Mix': 'high_scm', 'Manual': round(manual_2, 4),
     'compute_co2': round(app_2, 4), 'Diff': round(diff_2, 6), 'Pass': passed_2},
    {'Mix': 'minimal', 'Manual': round(manual_3, 4),
     'compute_co2': round(app_3, 4), 'Diff': round(diff_3, 6), 'Pass': passed_3},
])
print('CO2 validation summary:')
print(co2_results.to_string(index=False))
CO2_PASSED = int(co2_results['Pass'].sum())

CO2 validation summary:
                Mix   Manual  compute_co2  Diff  Pass
high_cement_low_scm 697.8553     697.8553   0.0  True
           high_scm 462.2553     462.2553   0.0  True
            minimal 503.0392     503.0392   0.0  True


---
## 6. Summary

In [14]:
summary = pd.DataFrame([
    {'Section': '2. Reproducibility',  'Tests Run': len(repro_df),      'Tests Passed': REPRO_PASSED},
    {'Section': '3. Edge Cases',       'Tests Run': len(edge_df),       'Tests Passed': EDGE_PASSED},
    {'Section': '4. SHAP Additivity',  'Tests Run': len(shap_check_df), 'Tests Passed': SHAP_PASSED},
    {'Section': '5. CO2 Calculations', 'Tests Run': len(co2_results),   'Tests Passed': CO2_PASSED},
])

summary['All Pass'] = summary['Tests Run'] == summary['Tests Passed']

print('VALIDATION RESULTS')
print(summary.to_string(index=False))

total_run = int(summary['Tests Run'].sum())
total_passed = int(summary['Tests Passed'].sum())
print(f'\nOverall: {total_passed} of {total_run} tests passed.')

VALIDATION RESULTS
            Section  Tests Run  Tests Passed  All Pass
 2. Reproducibility          3             3      True
      3. Edge Cases          5             5      True
 4. SHAP Additivity          2             2      True
5. CO2 Calculations          3             3      True

Overall: 13 of 13 tests passed.


### Results

- **Section 2 (Reproducibility):** Three rows from the cleaned dataset (low, mid, high strength) ran through the model and produced predictions. The printed input values and predictions are the reference points for manual cross-checks against the Streamlit app's Strength Predictor and Observation Explorer pages.
- **Section 3 (Edge Cases):** Five hand-crafted boundary mixes (high cement, high SCM, max age, min age, no supplementary materials) all produced predictions and SHAP top-3 features without error.
- **Section 4 (SHAP Additivity):** Per-feature SHAP values for the low and high samples reconstruct the model prediction within floating-point tolerance (`prediction ≈ base_value + sum(SHAP)`), confirming the explainer's outputs are mathematically consistent with the model itself.
- **Section 5 (CO₂ Calculations):** Three test mixes (high cement, high SCM, minimal) cross-check `compute_co2()` against transparent manual calculations using the documented `EMISSION_FACTORS` table. The function correctly skips non-material features (`temperature`, `age`) and zero-mass ingredients.

### Interpretations

The Streamlit app's three calculation surfaces — strength prediction, SHAP feature attribution, and CO₂ emissions — all reproduce values that are consistent with the trained model artifact and the documented emission factors. The validation notebook can be re-run anytime to confirm that future code changes (model refits, refactors, factor updates) do not silently alter app outputs.

**Manual verification step (out of scope for the notebook):** Take any row from the reproducibility section or any mix from the edge case section, enter its inputs into the app's slider widgets (rounded to slider step granularity), and confirm the displayed predicted strength, top SHAP features, and CO₂ value match the values printed here.

### Modeling Implications

This notebook validates the **app's wiring** to the trained model and emission factor table. It does not validate the underlying model accuracy (covered in notebook 02 via held-out test set RMSE) or the correctness of the published emission factors (sourced from peer-reviewed LCA literature; see `docs/emission_factors_v2.md`).

### Model Notes

- This notebook is **read-only**. It does not refit the model, modify any artifacts, or write any files.
- It is **self-contained**. Re-run top-to-bottom anytime as a regression check after changes to `app/emission_factors.py`, `app/shared.py`, or the model artifact at `data/results/xgb_tuned_model.joblib`.
- SHAP additivity tolerance is `1e-3` MPa, sufficient for floating-point noise from the `TreeExplainer` reconstruction.
- CO₂ tolerance is `0.01` kg CO₂/m³, sufficient for floating-point noise in the per-ingredient sum.